### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [ ]:
# incident_id|service|severity|duration_min|channel|acknowledged
rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|sms|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    # TODO: split строку по '|'
    parts = row.split("|")
    # TODO: убрать лишние пробелы у частей через strip()
    cleaned_parts = [part.strip() for part in parts]

    # TODO: ожидать 6 частей: incident_id, service, severity, duration_raw, channel, acknowledged_raw
    # TODO: если частей не 6 -> raise IncidentFormatError(...)
    if len(cleaned_parts) != 6:
        raise IncidentFormatError("Неверная строка")
    
    incident_id, service, severity, duration_raw, channel, acknowledged_raw = parts

    # TODO: duration_raw преобразовать в float
    try:
        duration = float(duration_raw)
    except ValueError:
        # TODO: при ошибке преобразования использовать raise DurationError(...) from exc
        raise DurationError(f"Не удалось преобразовать duration в число: {duration_raw}")
    
    # TODO: проверить, что duration > 0
    if duration <= 0:
        raise DurationError(f"Duration должен быть положительным числом, получено: {duration}")

    # TODO: проверить severity в {'low', 'medium', 'high', 'critical'}
    valid_severities = {'low', 'medium', 'high', 'critical'}
    if severity not in valid_severities:
        raise SeverityError(f"Недопустимый уровень severity: {severity}")

    # TODO: проверить channel в {'email', 'slack', 'pager'}
    valid_channel = {'email', 'slack', 'pager'}
    if channel not in valid_channel:
        raise ChannelError(f"Недопустимый канал: {channel}")

    # TODO: проверить acknowledged_raw в {'yes', 'no'}
    if acknowledged_raw == 'yes':
        acknowledged = True
    elif acknowledged_raw == 'no':
        acknowledged = False
    else:
        raise AcknowledgedFlagError(f"Недопустимое значение acknowledged: {acknowledged_raw}")
    # TODO: превратить acknowledged_raw в bool
    # TODO: вернуть словарь с разобранными полями
    return {
        'incident_id': incident_id,
        'service': service,
        'severity': severity,
        'duration_min': duration,
        'channel': channel,
        'acknowledged': acknowledged
    }

def process_batch(rows):
    # TODO: создать списки incidents и errors
    incidents = []
    errors = []

    # TODO: пройтись по rows циклом
    for row in rows:
        # TODO: внутри try вызвать parse_incident(row)
        try:
            incident = parse_incident(row)
            # TODO: валидный incident добавить в incidents
            incidents.append(incident)
            # TODO: IncidentProcessingError сохранить в errors как (row, error_type, message)
        except IncidentProcessingError as e:
            errors.append((row, type(e).__name__, str(e)))
    # TODO: вернуть (incidents, errors)
    return incidents, errors

# TODO: вызвать process_batch(rows)
incidents, errors = process_batch(rows)

# TODO: вывести количество валидных инцидентов и количество ошибок
print(f"Количество валидных инцидентов: {len(incidents)}")
print(f"Количество ошибок: {len(errors)}\n")

# TODO: собрать error_counts: dict[str, int]
error_count = {}
for _,error_type,_ in errors:
    if error_type in error_count:
        error_count[error_type] += 1
    else:
        error_count[error_type] = 1

# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
unacked_by_channel = {'email': [], 'slack': [], 'pager': []}
for incident in incidents:
    if not incident['acknowledged']:
        unacked_by_channel[incident['channel']].append(incident['incident_id'])

# TODO: собрать average_duration_by_severity только по валидным строкам
severity_durations = {}
for incident in incidents:
    severity = incident['severity']
    duration = incident['duration_min']

    if severity not in severity_durations:
        severity_durations[severity] = []

    severity_durations[severity].append(duration)

average_duraction_by_severity = {}
for severity, durations in severity_durations.items():
    average_duraction_by_severity[severity] = sum(durations) / len(durations)

# TODO: найти longest_incident среди валидных инцидентов по duration_min
if incidents:
    longest_incident = max(incidents, key=lambda x: x['duration_min'])
else:
    longest_incident = None

# TODO: красиво вывести получившиеся структуры
print(f"ID: {longest_incident['incident_id']}")
print(f"Сервис: {longest_incident['service']}")
print(f"Серьёзность: {longest_incident['severity']}")
print(f"Длительность: {longest_incident['duration_min']} мин")
print(f"Канал: {longest_incident['channel']}")
print(f"Подтверждён: {longest_incident['acknowledged']}")